# Matsci+20K Single Prediction Demo

This notebook loads one released NSF-SciFy matsci+20K LoRA adapter, runs it on one Hugging Face dataset entry, and evaluates that single prediction with the repository `evaluate_model` helper.

Default task: technical abstract -> non-technical abstract using `darpa-scify/nsf-scify-matsci-20k-nta`.

## Environment

Run this in the same GPU environment used for repository training/evaluation. The model cell requires CUDA and Unsloth. Uncomment and adjust the install command if the environment is not already set up.

In [ ]:
# Optional setup for a fresh environment. Unsloth installation can vary by CUDA/PyTorch version.
# %pip install datasets evaluate bert-score rouge-score sacrebleu python-dotenv
# See https://github.com/unslothai/unsloth for the matching Unsloth install command.

In [ ]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
for path in [REPO_ROOT / "src", REPO_ROOT / "scripts"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from path_utils import configure_hf_home, load_repo_env, resolve_root

load_repo_env()
REPO_ROOT = resolve_root(os.getenv("NSFSCIFY_ROOT_DIR", REPO_ROOT))
configure_hf_home(REPO_ROOT)
print("Repository root:", REPO_ROOT)
print("HF_HOME:", os.environ["HF_HOME"])

## Load One Combined Matsci+20K Test Entry

There is not a separate combined Hugging Face dataset repository, so the combined split is made by loading the released matsci and 20K datasets separately and concatenating the matching split.

In [ ]:
from datasets import concatenate_datasets, load_dataset

matsci = load_dataset("darpa-scify/nsf-scify-matsci")
twenty_k = load_dataset("darpa-scify/nsf-scify-20k")

combined_test = concatenate_datasets([matsci["test"], twenty_k["test"]])
example = combined_test[0]

print("Combined test size:", len(combined_test))
print("Award ID:", example.get("award_id"))
print("Title:", example.get("title"))
print("Technical abstract preview:")
print(example["technical_abstract"][:1000])

## Load The Released Matsci+20K NTA Adapter

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "darpa-scify/nsf-scify-matsci-20k-nta"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loaded:", MODEL_NAME)

## Format One Prompt

This uses the same prompt formatter as `--task tech2nontech --prompt_mode tech2nontech_instruct_user_assistant`.

In [ ]:
from datasets import Dataset
from data_utils import formatting_prompts_func_tech2nontech_instruct_user_assistant

formatted = formatting_prompts_func_tech2nontech_instruct_user_assistant(
    example,
    tokenizer=tokenizer,
    eval_mode=True,
)

single_example = dict(example)
single_example["text"] = formatted["text"]
single_dataset = Dataset.from_list([single_example])

print(single_dataset[0]["text"][:1200])

## Generate And Evaluate One Prediction

`evaluate_model` generates the prediction and computes BERTScore, ROUGE, and BLEU against the gold non-technical abstract for this one entry.

In [ ]:
from eval_utils import evaluate_model

results, preds, technical_abstracts, gold_non_technical_abstracts = evaluate_model(
    model,
    tokenizer,
    single_dataset,
    debug=False,
)

prediction = preds[0]
gold = gold_non_technical_abstracts[0]
metrics = results[0]

print("PREDICTION:\n")
print(prediction)
print("\nGOLD NON-TECHNICAL ABSTRACT:\n")
print(gold)
print("\nSINGLE-EXAMPLE METRICS:\n")
print(metrics)